In [1]:
import os
import re
import numpy as np
import faiss
import ollama

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

In [4]:
import os
import re

# 📝 Fichier principal
source_file = "assurance_complet.txt"

# 📁 Dossier où seront créés les sous-fichiers
output_folder = "data"
os.makedirs(output_folder, exist_ok=True)

# Lire le contenu du fichier principal
with open(source_file, "r", encoding="utf-8") as f:
    text = f.read()

# Diviser par sections principales : "# 1.", "# 2.", "# 3.", "# 4."
# On capture le titre et tout ce qui suit jusqu'au prochain "# <chiffre>"
sections = re.split(r"(# \d+\..*?)\n", text)

section_dict = {}

for i in range(1, len(sections), 2):
    title = sections[i].strip()
    content = sections[i+1].strip()

    # Nettoyage : enlever les "#" seuls
    content = re.sub(r"^\s*#\s*$", "", content, flags=re.MULTILINE).strip()

    # Nommer le fichier selon la section principale
    if "1." in title:
        filename = "assurance_garanties.txt"
    elif "2." in title:
        filename = "assurance_options.txt"
    elif "3." in title:
        filename = "assurance_packs.txt"
    elif "4." in title:
        filename = "assurance_services.txt"
    else:
        filename = "assurance_autre.txt"

    # Ajouter le contenu à la section
    if filename in section_dict:
        section_dict[filename] += "\n\n" + content
    else:
        section_dict[filename] = content

# Sauvegarder chaque sous-fichier
for fname, content in section_dict.items():
    path = os.path.join(output_folder, fname)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Fichier créé : {path}")

print("✅ Tous les sous-fichiers ont été créés dans le dossier 'data'")

Fichier créé : data\assurance_autre.txt
Fichier créé : data\assurance_garanties.txt
Fichier créé : data\assurance_options.txt
Fichier créé : data\assurance_packs.txt
Fichier créé : data\assurance_services.txt
✅ Tous les sous-fichiers ont été créés dans le dossier 'data'


In [5]:
import os

# Dossier où sont les sous-fichiers
input_folder = "data"

# Dossier où seront créés les chunks
output_folder = "chunks"
os.makedirs(output_folder, exist_ok=True)

CHUNK_SIZE = 400   # Nombre de caractères par chunk
OVERLAP = 80       # Chevauchement pour ne pas couper l'info importante

def chunk_text(text):
    """
    Découpe un texte en chunks avec chevauchement
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + CHUNK_SIZE
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start += CHUNK_SIZE - OVERLAP  # chevauchement
    return chunks

# Découper tous les fichiers et sauvegarder chaque chunk
for file in os.listdir(input_folder):
    if not file.endswith(".txt"):
        continue
    path = os.path.join(input_folder, file)
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    
    chunks = chunk_text(text)
    
    for i, chunk in enumerate(chunks):
        chunk_file = f"{file}_chunk{i+1}.txt"
        chunk_path = os.path.join(output_folder, chunk_file)
        with open(chunk_path, "w", encoding="utf-8") as f:
            f.write(chunk)

print("✅ Tous les fichiers ont été découpés en chunks dans le dossier 'chunks'")

✅ Tous les fichiers ont été découpés en chunks dans le dossier 'chunks'


In [6]:
import os

chunks_folder = "chunks"  # Dossier où sont les chunks
MAX_PREVIEW = 300          # Nombre de caractères à afficher pour chaque chunk

# Parcourir tous les fichiers du dossier chunks
for file in sorted(os.listdir(chunks_folder)):
    if not file.endswith(".txt"):
        continue
    path = os.path.join(chunks_folder, file)
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()
    
    print(f"\n=== {file} ===")
    print(content[:MAX_PREVIEW] + ("..." if len(content) > MAX_PREVIEW else ""))


=== assurance_autre.txt_chunk1.txt ===
MAGHREBIA est une compagnie d'assurance tunisienne reconnue, offrant une large gamme de couvertures adaptées aux besoins des particuliers et des familles. Ses principaux services incluent : assurance auto, santé, voyage, habitation, assurance scolaire, et bien d'autres produits personnalisés.

MAGHR...

=== assurance_autre.txt_chunk2.txt ===
honneur à fournir des solutions adaptées, un service client réactif et des offres modulables pour répondre aux besoins spécifiques de chaque foyer tunisien.

#  Assurance Habitation – Maghrebia

## Présentation générale
L’assurance habitation Maghrebia protège votre logement, que vous soyez locatair...

=== assurance_autre.txt_chunk3.txt ===
lle couvre les dommages aux biens, la responsabilité civile, ainsi que plusieurs services d’assistance et options supplémentaires pour mieux protéger votre foyer. L’objectif est d’assurer votre tranquillité et votre sécurité au quotidien.

## Objectif de l’assurance habit

In [7]:
import os

input_folder = "data"
output_folder = "chunks"

os.makedirs(output_folder, exist_ok=True)

CHUNK_SIZE = 400
OVERLAP = 80


def chunk_text(text):
    chunks = []
    start = 0
    while start < len(text):
        end = start + CHUNK_SIZE
        chunk = text[start:end]
        chunks.append(chunk)
        start += CHUNK_SIZE - OVERLAP
    return chunks


documents = []

for file in os.listdir(input_folder):

    # Ignorer les fichiers qui ne sont pas des .txt
    if not file.endswith(".txt"):
        continue

    path = os.path.join(input_folder, file)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):
        chunk_file = f"{file}_chunk{i}.txt"
        chunk_path = os.path.join(output_folder, chunk_file)
        with open(chunk_path, "w", encoding="utf-8") as f:
            f.write(chunk)

print("✅ Chunks créés")

✅ Chunks créés


In [8]:
# Fonction pour afficher les chunks par fichier
def show_chunks(files, folder="data", max_len=300):
    def split_text(text, max_len=max_len):
        sentences = text.split("\n")
        chunks = []
        chunk = ""
        for sentence in sentences:
            if len(chunk) + len(sentence) < max_len:
                chunk += sentence + "\n"
            else:
                chunks.append(chunk.strip())
                chunk = sentence + "\n"
        if chunk:
            chunks.append(chunk.strip())
        return chunks

    # Parcourir chaque fichier et afficher ses chunks
    for fname in files:
        path = os.path.join(folder, fname)
        with open(path, "r", encoding="utf-8") as f:
            content = f.read()
        
        chunks = split_text(content)
        print(f"\n=== Chunks de {fname} ===")
        for i, c in enumerate(chunks):
            print(f"\nChunk {i+1}:\n{c[:500]}")  # Affiche les 500 premiers caractères pour ne pas saturer

# Liste de fichiers que tu as déjà
files = [
    "assurance_garanties.txt",
    "assurance_options.txt",
    "assurance_packs.txt",
    "assurance_services.txt",
    "assurance_autre.txt"
]

# Appel de la fonction
show_chunks(files)



=== Chunks de assurance_garanties.txt ===

Chunk 1:
- Cambriolage
- Effraction
- Détérioration lors du vol
- Couverture des biens mobiliers et objets de valeur

- Fuite et rupture de canalisation
- Dommages aux installations sanitaires
- Frais de recherche de fuite

- Incendie
- Explosion / Implosion
- Foudre
- Dommages électriques et surtensions

Chunk 2:
- Fumée et dégâts connexes

- Tempêtes et intempéries
- Chute d’objets ou d’aéronefs
- Dommages matériels aux bâtiments

- Dommages corporels causés à un tiers
- Dommages matériels causés à un tiers
- Couverture pour les membres de la famille

- Assistance juridique en cas de sinistre

Chunk 3:
- Prise en charge des démarches de recours

- Objets personnels transportés lors de séjours
- Couverture en cas de vol ou dommage

=== Chunks de assurance_options.txt ===

Chunk 1:
Une équipe d’intervention est disponible 24h/24 pour :  
- Plomberie : fuites, rupture de canalisation  
- Électricité : coupure interne, court-circuit  
- Serrure

In [9]:
def clean_chunk(text):
    # Supprimer les hashtags et titres markdown
    text = text.replace("#", "")
    
    # Supprimer les tirets inutiles au début
    text = text.replace("- ", "")
    
    # Supprimer les multiples lignes vides
    text = "\n".join([line.strip() for line in text.splitlines() if line.strip()])
    
    return text

In [10]:

# 2️⃣ Charger et nettoyer tous les chunks
all_chunks = []
chunk_sources = []

for file in sorted(os.listdir(chunks_folder)):
    if not file.endswith(".txt"):
        continue
    path = os.path.join(chunks_folder, file)
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()
        cleaned = clean_chunk(content)
        all_chunks.append(cleaned)
        chunk_sources.append(file)

print(f"✅ {len(all_chunks)} chunks chargés et nettoyés\n")


✅ 20 chunks chargés et nettoyés



In [11]:
# 3️⃣ Affichage rapide des chunks nettoyés
for i, chunk in enumerate(all_chunks):
    preview = chunk[:MAX_PREVIEW] + ("..." if len(chunk) > MAX_PREVIEW else "")
    print(f"Chunk {i+1} ({chunk_sources[i]}): {preview}")


Chunk 1 (assurance_autre.txt_chunk0.txt): MAGHREBIA est une compagnie d'assurance tunisienne reconnue, offrant une large gamme de couvertures adaptées aux besoins des particuliers et des familles. Ses principaux services incluent : assurance auto, santé, voyage, habitation, assurance scolaire, et bien d'autres produits personnalisés.
MAGHRE...
Chunk 2 (assurance_autre.txt_chunk1.txt): honneur à fournir des solutions adaptées, un service client réactif et des offres modulables pour répondre aux besoins spécifiques de chaque foyer tunisien.
Assurance Habitation – Maghrebia
Présentation générale
L’assurance habitation Maghrebia protège votre logement, que vous soyez locataire ou pro...
Chunk 3 (assurance_autre.txt_chunk2.txt): lle couvre les dommages aux biens, la responsabilité civile, ainsi que plusieurs services d’assistance et options supplémentaires pour mieux protéger votre foyer. L’objectif est d’assurer votre tranquillité et votre sécurité au quotidien.
Objectif de l’assurance hab

In [12]:
print("Création des embeddings...")

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(all_chunks, convert_to_numpy=True)


Création des embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
print(f"\n✅ Embeddings créés : {embeddings.shape}")


✅ Embeddings créés : (20, 384)


In [14]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Index FAISS créé :", index.ntotal)


Index FAISS créé : 20


In [17]:
# 7 BM25 INDEX
# ==========================================

tokenized_chunks = [chunk.lower().split() for chunk in all_chunks]

bm25 = BM25Okapi(tokenized_chunks)


In [18]:
# 8 HYBRID SEARCH
# ==========================================

def hybrid_search(query, top_k=6):

    # FAISS

    query_vec = model.encode([query], convert_to_numpy=True)

    distances, indices = index.search(query_vec, top_k)

    faiss_results = [all_chunks[i] for i in indices[0]]

    # BM25

    tokenized_query = query.lower().split()

    bm25_scores = bm25.get_scores(tokenized_query)

    bm25_top = np.argsort(bm25_scores)[::-1][:top_k]

    bm25_results = [all_chunks[i] for i in bm25_top]

    combined = list(set(faiss_results + bm25_results))

    return combined

In [19]:
# 9 AI RERANKING
# ==========================================

def rerank_chunks(question, chunks):

    scored = []

    q_vec = model.encode([question], convert_to_numpy=True)

    for chunk in chunks:

        c_vec = model.encode([chunk], convert_to_numpy=True)

        score = np.dot(q_vec, c_vec.T)

        scored.append((score, chunk))

    scored.sort(reverse=True)

    return [c[1] for c in scored]


In [23]:
def chat_rag(model_name="llama3.1:8b", top_k=3):

    print("\n💬 Chatbot Assurance Maghrebia")
    print("Tape 'quit' pour sortir\n")

    history = []

    while True:

        question = input("Vous : ")

        if question.lower() in ["quit", "exit"]:

            print("👋 Fin du chatbot")

            break

        history.append({"role": "user", "content": question})

        # Hybrid Search

        results = hybrid_search(question, top_k * 2)

        # Re-ranking

        ranked = rerank_chunks(question, results)

        context = "\n".join(ranked[:top_k])

        # Mémoire courte

        conversation = ""

        for msg in history[-4:]:

            role = "Client" if msg["role"] == "user" else "Conseiller"

            conversation += f"{role} : {msg['content']}\n"

        # Prompt

        prompt = f"""
Tu es un assistant virtuel expert en assurance pour la compagnie MAGHREBIA (Tunisie).

Ta mission est d'aider les clients à comprendre l'assurance habitation.

Règles :

1️⃣ Utilise d'abord les informations du CONTEXTE.
2️⃣ Si le CONTEXTE ne contient pas toute l'information, tu peux compléter avec ta connaissance générale sur l'assurance habitation.
3️⃣ Quand tu complètes avec ton savoir, précise que c'est une information générale.
4️⃣ Ne donne jamais d'informations absurdes ou inventées.
5️⃣ Réponds comme un conseiller client professionnel.

CONVERSATION :
{conversation}

CONTEXTE :
{context}

QUESTION CLIENT :
{question}

REPONSE CONSEILLER :
"""

        response = ollama.chat(

            model=model_name,

            options={"temperature": 0.1},

            messages=[
                {"role": "user", "content": prompt}
            ]

        )

        answer = response["message"]["content"]

        history.append({"role": "assistant", "content": answer})

        print("\n🤖 Chatbot :", answer, "\n")


In [24]:
chat_rag()


💬 Chatbot Assurance Maghrebia
Tape 'quit' pour sortir



Vous :  parle moi de asusrance de vol



🤖 Chatbot : Bonjour ! Je suis ravi de vous aider à comprendre l'assurance de vol. Cependant, je remarque que le contexte fourni ne mentionne pas spécifiquement l'assurance de vol. Mais je peux vous donner des informations générales sur ce type d'assurance.

L'assurance de vol est un type d'assurance qui couvre les biens volés ou endommagés dans votre habitation. Elle peut inclure les biens personnels, tels que les bijoux, les objets de valeur, les vêtements, les meubles, etc.

Avec l'assurance de vol, vous bénéficierez d'une couverture financière en cas de vol ou d'endommagement de vos biens. Vous pourrez ainsi récupérer les sommes nécessaires pour remplacer ou réparer vos biens.

Il est important de noter que l'assurance de vol peut varier en fonction de la compagnie d'assurance et des conditions de votre contrat. Il est donc recommandé de vérifier les détails de votre contrat pour en savoir plus sur les couvertures et les exclusions.

Si vous souhaitez en savoir plus sur l'assurance

Vous :  assurance maghribeya



🤖 Chatbot : Bonjour ! Je suis ravi de vous aider à comprendre l'assurance Maghrebia. En tant que compagnie d'assurance tunisienne reconnue, Maghrebia offre une large gamme de couvertures adaptées aux besoins des particuliers et des familles.

Puisque vous avez mentionné l'assurance Maghrebia, je suppose que vous souhaitez en savoir plus sur l'assurance habitation proposée par cette compagnie. L'assurance habitation Maghrebia protège votre logement, que vous soyez locataire ou propriétaire, et couvre les dommages aux biens, la responsabilité civile, ainsi que plusieurs autres risques.

Si vous avez des questions spécifiques sur l'assurance habitation Maghrebia ou si vous souhaitez en savoir plus sur les conditions et les couvertures proposées, n'hésitez pas à me les poser. Je serai ravi de vous aider ! 



Vous :  quit


👋 Fin du chatbot


In [25]:
import pickle
import numpy as np
import faiss
import json
import os

os.makedirs("saved_data", exist_ok=True)

# 1️⃣ Chunks et sources
with open("saved_data/all_chunks.pkl", "wb") as f:
    pickle.dump(all_chunks, f)

with open("saved_data/chunk_sources.pkl", "wb") as f:
    pickle.dump(chunk_sources, f)

# 2️⃣ Embeddings
np.save("saved_data/embeddings.npy", embeddings)

# 3️⃣ Index FAISS
faiss.write_index(index, "saved_data/faiss_index.index")

# 4️⃣ Hyperparamètres
params = {
    "CHUNK_SIZE": 400,
    "OVERLAP": 80,
    "TOP_K": 3,
    "embedding_model": "all-MiniLM-L6-v2"
}
with open("saved_data/params.json", "w") as f:
    json.dump(params, f)